# Extremly randomized trees
At this point we have told about decision trees, a simple, powerful and fast approach for classification and regression tasks. However, can't fit complex relatiopnships among variables and tends to overfitting specially in deep trees. Random forests come to the rescue adding a manner to combine a set of decision trees for learning complex relationships and having a good performance, on implementing parallelization for decision trees training. Nevertheless random forests are slow when many decision trees must be trained in big datasets. We have a weak learner that is fast but can't learn complex topics, in the other hand we have a strong learner that is slow. Here is when extremly randomized trees appears offering a balanced way between performance and training velocity.

## Definition
Extremly randomized trees (aka extratrees) is a supervised ML algorithm based on decision trees technique, as the name says the main feature is related with taking randomness to the extrem. An extratree is similar to a decision tree, its components are equals, the process to evaluate an instance, split criterions, etc. There is only a small but powerful change in the decision trees building algorithm that makes this approach faster and sometimes better than random forests and decision trees. Before going in deep with the key difference from extratrees to decision trees, let's state the algorithm to build a decision tree from the definition porpoused in [decision trees notebook](https://www.kaggle.com/code/criser2013/decision-trees/notebook):

> 1. Split your dataset considering different features and their threshold value (that's the value used for comparing in the condition, for instance, if condition is `sqrt_feets > 9`, `9` is the threshold and `sqrt_feets` is the feature).
> 2. Calculate a metric like Gini impurity or information gain to each split.
> 3. Select the best split based on the metric.
> 4. Create 2 (or more for categorical variables) sub-splits based on the selected condition and pass them to the corresponding child node.
> 5. Repeat this procedure in the children until reaching a stop criterion (max depth is reached, no splits left, etc).

## How does differs from decision trees?
Extratrees modifies step 1, while decision trees follows a deterministic algorithm that builds any possible threshold value for each variable, extratrees just selects one random feature. That's the main advantage of this technique, build decision trees faster and thanks to the random approach, variance is lower than always finding the best split. This is a high level explanation, so let's go in deep in how each algorithm finds its threshold candidates and how expensive are in computational costs.

### Selecting the best sample split in decision trees
1. For a given feature $i$, sort every unique value ${v_i}_{k}$ following an ascending order.
<center>
    ${v_i}_{1} < {v_i}_{2} < ... < {v_i}_{k}$
</center>

2. Calculate the thresholds from median points between every unique value pair.
<center>
    ${t_i}_{k} = \frac{{v_i}_{k} + {v_i}_{k+1}}{2}$
</center>

3. Divide node samples in the following subsets: $D_{left} = \{x \in D | {x_j}_{i} \leq {t_i}_{k} \}$ and $D_{right} = \{x \in D | {x_j}_{i} > {t_i}_{k} \}$

The algorithm belongs to $O(nlog(n) + nk)$ class assuming a sorting algorithm like quicksort, but when a feature has $n$ unique values (each single instance has an unique value) $k = n - 1$, so the algorithm belongs to $O(n^{2})$ class.

### Selecting the best sample split in extratrees
1. For a given feature $i$, get the minimum ${v_i}_{min}$ and maximum ${v_i}_{max}$ value.
2. Choose a random threshold value ${t_i}_{k}$ from range $[{v_i}_{min}, {v_i}_{max}]$ assuming a continuous (discrete for categorical variables) uniform distribution.
3. Divide node samples in the following subsets: $D_{left} = \{x \in D | {x_j}_{i} \leq {t_i}_{k} \}$ and $D_{right} = \{x \in D | {x_j}_{i} > {t_i}_{k} \}$.

Thanks to random selection the algorithm belongs to $O(n)$ class, because the most expensive steps are finding bounds and dividing the node samples, so both consists on a single comparisson of every data instance in our dataset.

## How does differs from random forests?
Extratrees have an ensemble implementation similara to random forests, as we mentioned before, key difference with decision trees is the replacement of a deterministic strategy to choose the best split on each node. As random forests, the ensemble approach consists on training many extratrees but **without using bootstrapping** to generate random samples for each tree. In this case every tree is trained with the **whole dataset** because random node splits accomplish the same goal of bootstrapping, reduce variance at each tree.

## Implementation
Classification and regression tasks works as the same way as decision trees, so I won't go in deep. I would like to notice that Scikit library has `ExtraTreeClassifier` and `ExtraTreeRegressor` for training a single tree and `ExtraTreesClassifier` and `ExtraTreesRegressor` for the ensemble approach.

In [1]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = ExtraTreesClassifier(n_estimators=20, criterion="entropy", max_depth=10, random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

Accuracy: 81.750000 %
Weighted F1-Score: 81.373105 %


In [2]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import ExtraTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = ExtraTreeRegressor(criterion="squared_error", random_state=123)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

Mean squared error: 1712235348.274637
Mean absolute error: 6907.316468


## Most relevant hyperparameters
- **`n_estimators`:** Number of trees in the forest (only in `ExtraTreesClassifier` and `ExtraTreesRegressor` classes).
- **`splitter`:** Mechanism to select the best split (only in `ExtraTreeClassifier` and `ExtraTreeRegressor` classes).
- **`criterion`:** Function used to measure the quality of a split (`gini`, `entropy`, `mae`, `mse`, etc).
- **`max_depth`:** Maximum depth of the tree, if not specified, nodes are expanded until leaves are pure or all leaves contains less than `min_samples_split` samples.
- **`min_samples_split`:** Minimum number of samples required to split an internal node.
- **`min_samples_leaf`:** Minimun number of samples required to be a leaf node.
- **`max_features`:** The number of features to consider when looking the best split.
- **`boostrap`:** A flag for disabling bootstrapping (only in `ExtraTreesClassifier` and `ExtraTreesRegressor` classes).